In [1]:
## Importing Libraries
import numpy as np
import pandas as pd

In [2]:
### Reading CSV Files
df1 = pd.read_csv('gloss_counts.csv') #this is the original 147 glosses from our cleaned dataset
df1 = df1[['gloss']].rename(columns={'gloss': 'Gloss'})
df1.head()

,Gloss
0,1 DOLLAR
1,5 DOLLARS
2,8 HOUR
3,9 OCLOCK
4,A LINE BOB


In [3]:
len(df1)

147

In [4]:
top_250_df = pd.read_csv('asl_top_250_essential.csv') ##this is the collected 250 common and important gloss
top_250_df = top_250_df[['Gloss']]
top_250_df.head()

,Gloss
0,ME
1,YOU
2,HE
3,SHE
4,IT


In [5]:
len(top_250_df)

250

In [6]:
df_103_research = pd.read_csv('sorted_mean_rating.csv') #this is the gloss from the research paper
df_103_research = df_103_research[0:103]
df_103_research = df_103_research[['Sign']].rename(columns={'Sign': 'Gloss'})
df_103_research.head()

,Gloss
0,Time
1,Home
2,Night
3,Internet
4,Drink


In [7]:
len(df_103_research)

103

In [8]:
### helper function to normalize
for df in [top_250_df, df1, df_103_research]:
    df['Gloss'] = df['Gloss'].str.strip().str.lower()

In [9]:
### check internal duplicates
print("top_250 duplicates:", top_250_df['Gloss'].duplicated().sum())
print("df1 duplicates:", df1['Gloss'].duplicated().sum())
print("df_103 duplicates:", df_103_research['Gloss'].duplicated().sum())

top_250 duplicates: 1
df1 duplicates: 0
df_103 duplicates: 0


In [10]:
top_250_df = top_250_df.drop_duplicates(subset=['Gloss'])
print("top_250 duplicates:", top_250_df['Gloss'].duplicated().sum())

top_250 duplicates: 0


In [11]:
### check overlaps between ALL files
common_1 = set(top_250_df['Gloss']) & set(df1['Gloss'])
common_2 = set(top_250_df['Gloss']) & set(df_103_research['Gloss'])
common_3 = set(df1['Gloss']) & set(df_103_research['Gloss'])

all_common = common_1 | common_2 | common_3

print("Total overlapping glosses:", len(all_common))

Total overlapping glosses: 80


In [12]:
### drop ALL overlaps from top_250 (we treat it as the main file)
# keep in df1 first
df_103_research = df_103_research[~df_103_research['Gloss'].isin(df1['Gloss'])]

top_250_df = top_250_df[
    ~top_250_df['Gloss'].isin(df1['Gloss']) &
    ~top_250_df['Gloss'].isin(df_103_research['Gloss'])
]

In [13]:
### check overlaps between ALL files
common_1 = set(top_250_df['Gloss']) & set(df1['Gloss'])
common_2 = set(top_250_df['Gloss']) & set(df_103_research['Gloss'])
common_3 = set(df1['Gloss']) & set(df_103_research['Gloss'])

all_common = common_1 | common_2 | common_3

print("Total overlapping glosses:", len(all_common))

Total overlapping glosses: 0


In [14]:
### merge everything
merged_df = pd.concat([top_250_df, df_103_research, df1], ignore_index=True)

In [15]:
merged_df.head()

,Gloss
0,he
1,she
2,it
3,we
4,they


In [16]:
len(merged_df)

407

In [17]:
print("merged_df duplicates:", merged_df['Gloss'].duplicated().sum())

merged_df duplicates: 0


In [18]:
### COMPENSATE to reach exactly 500
current_len = len(merged_df)
print("Before fill:", current_len)

if current_len < 500:
    df_ref = pd.read_csv('sorted_mean_rating.csv')
    df_ref = df_ref[['Sign']].rename(columns={'Sign': 'Gloss'})
    df_ref['Gloss'] = df_ref['Gloss'].str.strip().str.lower()

    needed = 500 - current_len

    fill_pool = df_ref[~df_ref['Gloss'].isin(merged_df['Gloss'])]

    extra = fill_pool.iloc[:needed]

    merged_df = pd.concat([merged_df, extra], ignore_index=True)

Before fill: 407


In [19]:
### final safety check
merged_df = merged_df.drop_duplicates(subset=['Gloss'])
print("Final length:", len(merged_df))

Final length: 500


In [20]:
 merged_df['Gloss'].duplicated().sum()

np.int64(0)

In [21]:
df_additional=pd.read_csv('Additional_Gloss_List.csv')
df_additional.head()

,Gloss
0,ANXIOUS
1,AWKWARD
2,BORROW
3,BRING
4,BUSY


In [22]:
len(df_additional)

71

In [23]:
print("df_additional duplicates:", df_additional['Gloss'].duplicated().sum())

df_additional duplicates: 0


In [24]:
### check overlaps between ALL files
common = set(merged_df['Gloss']) & set(df_additional['Gloss'])
print("Total overlapping glosses:", len(all_common))

Total overlapping glosses: 0


In [25]:
### merge everything
merged_df_final = pd.concat([merged_df,df_additional], ignore_index=True)
merged_df_final.head()

,Gloss
0,he
1,she
2,it
3,we
4,they


In [26]:
len(merged_df_final)

571

In [27]:
print("merged_df_final duplicates:", merged_df_final['Gloss'].duplicated().sum())

merged_df_final duplicates: 0


In [28]:
merged_df_final['Gloss'] = (
    merged_df_final['Gloss']
    .str.replace(r"\s*\(.*?\)", "", regex=True)  # remove ( ... )
    .str.replace("-", " ")                        # replace - with space
    .str.replace("_", " ")                        # replace _ with space
    .str.strip()                                  # clean spaces
    .str.upper()                                  # capitalize all
)

In [29]:
merged_df_final[merged_df_final['Gloss'].str.contains(r"/", na=False)]

,Gloss
178,FOOD/EAT
471,TYPE/KIND
481,MISS/GONE


In [30]:
merged_df_final = merged_df_final.assign(
    Gloss=merged_df_final['Gloss'].str.split('/')
).explode('Gloss')

merged_df_final['Gloss'] = merged_df_final['Gloss'].str.strip()

In [31]:
merged_df_final.head()

,Gloss
0,HE
1,SHE
2,IT
3,WE
4,THEY


In [32]:
merged_df_final[merged_df_final['Gloss'].str.contains(r"/", na=False)]

,Gloss


In [33]:
len(merged_df_final)

574

In [36]:
merged_df_final[merged_df_final["Gloss"].str.strip() == "$5"]

,Gloss
483,$5


In [37]:
merged_df_final = merged_df_final[
    merged_df_final["Gloss"].str.strip() != "$5"
]

In [38]:
merged_df_final[merged_df_final["Gloss"].str.strip() == "$5"]

,Gloss


In [39]:
len(merged_df_final)

573

In [41]:
### save
merged_df_final.to_csv('573_Gloss_List.csv', index=False)